# TP — Échangeur thermique : action intégrale par forwarding

**Commande non linéaire — Master 3EA, parcours ASI — Université Claude Bernard Lyon 1**

Ce notebook accompagne l'énoncé `TP_Echangeur.pdf`. Il **est** votre rapport : répondez aux questions dans les cellules de texte, commentez chaque figure.

> **Sur Colab :** commencez par *Fichier > Enregistrer une copie dans Drive*.

Les cellules marquées `# À COMPLÉTER` sont à écrire.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.figsize'] = (11, 3.4)

## 0. Le modèle (fourni)

$x = (T_1, T_2, T_3, \bar T_1, \bar T_2, \bar T_3)$ ; le fluide chaud entre dans le compartiment 1, le fluide froid dans le compartiment 3. La sortie régulée est $\bar T_{out} = \bar T_1 = Cx$. La commande est le débit chaud $q \in [0,\ q_M]$.

$$\dot x = A x + (B x + b_1 T_{in})\,q + (\bar B x + \bar b_1 \bar T_{in})\,\bar q .$$

In [ ]:
alpha, S = 500.0, 0.1                      # coefficient global d'échange (W/m²/K), surface (m²)
rho, V, cp = 997.0, 0.002, 4185.0          # eau, volume d'un compartiment
qbar, qM = 0.02, 0.05                      # débit froid, débit chaud maximal (kg/s)
T_in, Tb_in = 360.0, 300.0                 # températures d'entrée (K)

kth = alpha * S / (rho * V * cp)
A = kth * np.array([[-1, 0, 0, 1, 0, 0], [0, -1, 0, 0, 1, 0], [0, 0, -1, 0, 0, 1],
                    [1, 0, 0, -1, 0, 0], [0, 1, 0, 0, -1, 0], [0, 0, 1, 0, 0, -1]])
b1 = np.array([1, 0, 0, 0, 0, 0]) / (rho * V)
B = np.array([[-1, 0, 0, 0, 0, 0], [1, -1, 0, 0, 0, 0], [0, 1, -1, 0, 0, 0],
              [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]]) / (rho * V)
bb1 = np.array([0, 0, 0, 0, 0, 1]) / (rho * V)
Bb = np.array([[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0],
               [0, 0, 0, -1, 1, 0], [0, 0, 0, 0, -1, 1], [0, 0, 0, 0, 0, -1]]) / (rho * V)
C = np.array([0, 0, 0, 1, 0, 0.0])

def echangeur(x, q, Tb=Tb_in):
    return A @ x + (B @ x + b1 * T_in) * q + (Bb @ x + bb1 * Tb) * qbar

def equilibre(q, Tb=Tb_in):
    """État stationnaire pour un débit chaud constant q."""
    return -np.linalg.solve(A + q * B + qbar * Bb, b1 * T_in * q + bb1 * Tb * qbar)

def Tb_entree(t, perturbations):
    """perturbations : liste de (instant, valeur de Tb_in) ; Tb_in nominal avant le premier instant."""
    val = Tb_in
    for t0, v in perturbations:
        if t >= t0:
            val = v
    return val

# Séance 1 — Modélisation, point de fonctionnement, approche linéaire

### Q4. Boucle ouverte et caractéristique statique

In [ ]:
fig, ax = plt.subplots(1, 2)
for q in (0.0, 0.01, 0.025, 0.05):
    sol = solve_ivp(lambda t, x: echangeur(x, q), (0, 1500), 300 * np.ones(6), max_step=5.0)
    ax[0].plot(sol.t, C @ sol.y, label='q = %.3f kg/s' % q)
ax[0].set(xlabel='t (s)', ylabel=r'$\bar T_{out}$ (K)', title='réponses à débit constant'); ax[0].legend(fontsize=8)
# À COMPLÉTER : sortie stationnaire C x*(q) pour q dans [0, qM]
qs_grille, Tout_grille = [], []
ax[1].plot(qs_grille, Tout_grille); ax[1].set(xlabel='q (kg/s)', ylabel=r'$\bar T_{out}$ stationnaire (K)', title='caractéristique statique')
for a in ax:
    a.grid(True)
plt.tight_layout(); plt.show()

*Votre commentaire (quelles températures de sortie sont atteignables ?) :*

### Q5. Point de fonctionnement et sensibilité à $\bar T_{in}$

In [ ]:
q_s = 0.025
# À COMPLÉTER : x_s, consigne r = C x_s, et variation de la sortie si Tb_in augmente de 5 K (q = q_s)
x_s, r, ecart = None, None, None
print('x* =', x_s, '\nr = %.2f K ; effet de Tb_in + 5 K sur la sortie : %+.2f K' % (r, ecart))

### Q3 et Q6. Coordonnées centrées et hypothèse de Lyapunov
$\chi = x - x^*$, $u = q - q^*$ : $\dot\chi = A^*\chi + g(\chi)\,u$ avec $A^* = A + q^*B + \bar q\bar B$ et $g(\chi) = B(\chi + x^*) + b_1T_{in}$.

In [ ]:
# À COMPLÉTER
A_s = None
print('valeurs propres de A* :', np.linalg.eigvals(A_s))
print('valeurs propres de A* + A*^T (V = |chi|^2 convient si toutes < 0) :', np.linalg.eigvalsh(A_s + A_s.T))

### Q7. Une approche classique : PI réglé autour du point de fonctionnement
$q = q^* - k_p(\bar T_{out} - r) - k_i z$, avec $\dot z = \bar T_{out} - r$, et la saturation $0 \le q \le q_M$.

In [ ]:
def simule(loi, perturbations=(), T=4000.0, r_cible=None):
    """loi(x, z) -> q (non saturé) ; z intègre C x - r_cible."""
    rc = r if r_cible is None else r_cible
    def f(t, X):
        x, z = X[:6], X[6]
        q = np.clip(loi(x, z), 0.0, qM)
        return np.r_[echangeur(x, q, Tb_entree(t, perturbations)), C @ x - rc]
    sol = solve_ivp(f, (0, T), np.r_[x_s, 0.0], max_step=2.0, rtol=1e-8, atol=1e-8)
    q = np.array([np.clip(loi(sol.y[:6, j], sol.y[6, j]), 0, qM) for j in range(sol.y.shape[1])])
    return sol.t, C @ sol.y[:6], q

def loi_pi(kp=0.004, ki=2e-5, q_ff=None, r_cible=None):
    qf = q_s if q_ff is None else q_ff
    rc = r if r_cible is None else r_cible
    return lambda x, z: qf - kp * (C @ x - rc) - ki * z

def trace(resultats, titre, ref=None):
    fig, ax = plt.subplots(1, 2)
    for lab, t, y, q in resultats:
        ax[0].plot(t, y, label=lab); ax[1].plot(t, q, label=lab)
    if ref is not None:
        ax[0].axhline(ref, color='k', ls=':', lw=0.8)
    ax[0].set(xlabel='t (s)', ylabel=r'$\bar T_{out}$ (K)', title=titre)
    ax[1].set(xlabel='t (s)', ylabel='q (kg/s)')
    for a in ax:
        a.grid(True); a.legend(fontsize=8)
    plt.tight_layout(); plt.show()

trace([('Tb_in + 5 K', *simule(loi_pi(), [(200, 305.0)])),
       ('Tb_in - 10 K', *simule(loi_pi(), [(200, 290.0)]))], 'PI', ref=r)

*Votre commentaire :*

# Séance 2 — Forwarding

### Q12. La fonction $M$ et la condition $L_gM(0)\neq0$
L'EDP $L_fM = h$ avec $f(\chi) = A^*\chi$, $h(\chi) = C\chi$ admet une solution linéaire $M(\chi) = M\chi$.

In [ ]:
# À COMPLÉTER : M (vecteur ligne) et L_g M(0)
M, LgM0 = None, None
print('M =', M)
print('L_g M(0) = %.1f' % LgM0)

### Q12 (suite). La loi de forwarding
$W = \ell\,|\chi|^2 + \tfrac p2\,(z - M\chi)^2$ et $u = -\ell\,L_gV + p\,(z - M\chi)\,L_gM$, avec $L_gV = 2\chi^\top g(\chi)$, $L_gM = M g(\chi)$.

In [ ]:
def loi_fwd(l=5e-5, p=3e-7, x_eq=None, q_eq=None, M_eq=None):
    xe = x_s if x_eq is None else x_eq
    qe = q_s if q_eq is None else q_eq
    Me = M if M_eq is None else M_eq
    def loi(x, z):
        # À COMPLÉTER : renvoyer q = q* + u
        return qe
    return loi

for nom, pert in (('Tb_in + 5 K', [(200, 305.0)]), ('Tb_in - 10 K', [(200, 290.0)])):
    trace([('forwarding', *simule(loi_fwd(), pert)), ('PI', *simule(loi_pi(), pert))], nom, ref=r)

*Votre commentaire :*

### Q13. Saturation : $W$ décroît-elle encore ?
Démarrage à froid : tout l'échangeur est à 300 K. La commande sature au début. Complétez `W_et_Wpoint`, qui évalue $W$ et $\dot W$ le long de la trajectoire, avec la commande réellement appliquée (saturée).

In [ ]:
def W_et_Wpoint(x, z, l=5e-5, p=3e-7):
    chi = x - x_s; g = B @ x + b1 * T_in
    zeta = z - M @ chi
    u = np.clip(q_s - l * 2 * chi @ g + p * zeta * (M @ g), 0, qM) - q_s
    # À COMPLÉTER : W et sa dérivée Wp le long du système, avec la commande saturée u (formules de la question 10)
    W, Wp = 0.0, 0.0
    return W, Wp

sol = solve_ivp(lambda t, X: np.r_[echangeur(X[:6], np.clip(loi_fwd()(X[:6], X[6]), 0, qM)), C @ X[:6] - r],
                (0, 3000), np.r_[300 * np.ones(6), 0.0], max_step=2.0, rtol=1e-8, atol=1e-8)
WW = np.array([W_et_Wpoint(sol.y[:6, j], sol.y[6, j]) for j in range(sol.y.shape[1])])
qq = np.array([np.clip(loi_fwd()(sol.y[:6, j], sol.y[6, j]), 0, qM) for j in range(sol.y.shape[1])])
print('commande saturée pendant %.0f %% des pas de temps ; max de dW/dt = %.2e' % (100 * np.mean((qq <= 0) | (qq >= qM)), WW[:, 1].max()))
fig, ax = plt.subplots(1, 3)
ax[0].plot(sol.t, C @ sol.y[:6]); ax[0].axhline(r, color='k', ls=':'); ax[0].set(xlabel='t (s)', ylabel=r'$\bar T_{out}$ (K)')
ax[1].plot(sol.t, qq); ax[1].set(xlabel='t (s)', ylabel='q (kg/s)')
ax[2].semilogy(sol.t, WW[:, 0]); ax[2].set(xlabel='t (s)', ylabel='W')
for a_ in ax:
    a_.grid(True)
plt.tight_layout(); plt.show()

*Votre commentaire : que montre ce calcul, et que ne montre-t-il pas ?*

### Q14. Changer de consigne
Pour une nouvelle consigne $r$, il faut recalculer $(q^*, x^*, M)$. La fonction ci-dessous le fait (fournie).

In [ ]:
def point_de_fonctionnement(r_cible, Tb=Tb_in):
    q = brentq(lambda q: C @ equilibre(q, Tb) - r_cible, 1e-6, qM)
    x = equilibre(q, Tb)
    Ms = C @ np.linalg.inv(A + q * B + qbar * Bb)
    return q, x, Ms

for r_new in (320.0, 338.0):
    qn, xn, Mn = point_de_fonctionnement(r_new)
    trace([('forwarding', *simule(loi_fwd(x_eq=xn, q_eq=qn, M_eq=Mn), r_cible=r_new)),
           ('PI', *simule(loi_pi(q_ff=qn, r_cible=r_new), r_cible=r_new))], 'consigne %.0f K' % r_new, ref=r_new)

*Votre commentaire.* Que se passe-t-il pour une consigne de 342 K ? (Essayez, puis expliquez avec la caractéristique statique.)

In [ ]:
# Votre essai pour r = 342 K

### Question ouverte (une au choix, obligatoire)
Choisissez une des trois questions de l'énoncé (réglage de $\ell$ et $p$, variation du débit froid, mesure de la seule sortie) et traitez-la ici. Elle n'est pas guidée et doit apparaître dans la soutenance.

In [ ]:
# Votre travail